# 01 Data Audit

## Business Question

原始数据的范围、字段、日期、用户覆盖和业务粒度是否足以支持截至2015-10-16的增长分析？

## Analysis Objective

复核Phase 2已经完成的原始数据审计，确认Members主键、注册日期、来源字段、用户日日志粒度、负播放时长和交易表可用性。本Notebook读取已生成审计产物，不重新扫描大型原始日志。

## Data Used

- `outputs/data_audit_table_summary.csv`
- `outputs/data_audit_2015-07-01_2015-10-16.json`
- `outputs/user_growth_profile_validation.json`

## Key Metrics

- 表行数、字段、日期范围、唯一用户数和数据粒度。
- Members：重复`msno`、无效注册日期、缺失`registered_via`。
- Logs：用户覆盖、负`total_secs`、缺失值、重复`msno+date`。
- Transactions：日期及用户覆盖，仅确认辅助可用性。


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
AS_OF_DATE = pd.Timestamp("2015-10-16")
ANALYSIS_START = pd.Timestamp("2015-07-01")
pd.set_option("display.max_columns", 50)


In [2]:
audit_summary = pd.read_csv(OUTPUTS_DIR / "data_audit_table_summary.csv")
audit = json.loads((OUTPUTS_DIR / "data_audit_2015-07-01_2015-10-16.json").read_text(encoding="utf-8"))
profile_validation = json.loads((OUTPUTS_DIR / "user_growth_profile_validation.json").read_text(encoding="utf-8"))
display(audit_summary)


,table,rows_full,rows_window,date_field,date_min,date_max,unique_users_full,unique_users_window,grain
0,members_v3.csv,6769473,504684,registration_init_time,2004-03-26,2017-04-29,6769473,504684,one row per registered user (msno)
1,user_logs.csv,392106543,47544997,date,2015-01-01,2017-02-28,5234111,1359381,one row per user-date (msno + date)
2,transactions.csv,21547746,2543772,transaction_date,2015-01-01,2017-02-28,2363626,896191,one row per transaction event; multiple rows p...


## Table Structure and Grain

- `members_v3.csv`：一行一个注册用户，`msno`为用户主键。
- `user_logs.csv`：一行一个用户日，业务键为`msno + date`。
- `transactions.csv`：一行一笔交易，同一用户同日可以存在多笔交易。


In [3]:
quality_rows = []
for table_name, table in audit["tables"].items():
    for check, value in table.get("quality", {}).items():
        quality_rows.append({"table": table_name, "check": check, "value": value})
quality_checks = pd.DataFrame(quality_rows)
display(quality_checks)


,table,check,value
0,members_v3.csv,missing_msno,0
1,members_v3.csv,duplicate_msno_rows_beyond_first,0
2,members_v3.csv,duplicate_msno_users,0
3,members_v3.csv,invalid_registration_dates,0
4,members_v3.csv,missing_registered_via,0
5,user_logs.csv,missing_msno_full,None
6,user_logs.csv,negative_total_secs_full,61493
7,user_logs.csv,negative_total_secs_window,40621
8,user_logs.csv,missing_total_secs_full,0
9,user_logs.csv,missing_total_secs_window,0


## User-growth Profile Validation

底表必须保持一行一个`msno`；未成熟行为指标为NULL，成熟但未发生行为才为0。


In [4]:
profile_validation_summary = pd.DataFrame({
    "check": ["rows", "unique_msno", "duplicate_msno", "registration_date_min", "registration_date_max",
              "negative_total_secs_set_to_zero", "active_days_out_of_range",
              "negative_output_play_count", "negative_output_total_secs"],
    "value": [profile_validation[k] for k in ["rows", "unique_msno", "duplicate_msno",
              "registration_date_min", "registration_date_max", "negative_total_secs_set_to_zero",
              "active_days_out_of_range", "negative_output_play_count", "negative_output_total_secs"]]
})
display(profile_validation_summary)
assert profile_validation["rows"] == profile_validation["unique_msno"]
assert profile_validation["duplicate_msno"] == 0
assert all(v["non_null_when_ineligible"] == 0 and v["null_when_eligible"] == 0
           for v in profile_validation["null_rule_checks"].values())


,check,value
0,rows,504684
1,unique_msno,504684
2,duplicate_msno,0
3,registration_date_min,2015-07-01
4,registration_date_max,2015-10-16
5,negative_total_secs_set_to_zero,155
6,active_days_out_of_range,0
7,negative_output_play_count,0
8,negative_output_total_secs,0


## Data-quality Conclusion

三张保留表可用于后续分析。Members主键、注册日期及来源字段通过检查；目标窗口日志无重复用户日。负`total_secs`只在时长累计时按0处理，不删除对应活跃日志。Transactions仅保留辅助用途。该结论不涉及新增异常或用户质量判断。
